# 11 — Masked vs unmasked: full metrics comparison

Loads the saved predictions for all five models, masked and unmasked (same seed each), and builds the full comparison table: accuracy, Macro-F1, and per-class recall (with Severe and Proliferate called out). Matched by seed, without-handling arm, so masking is the only variable.

**Attach:** the committed outputs of 10a–10e (masked preds) AND 03a/03b/03b2 (unmasked preds). No GPU.

In [1]:
import os, re, numpy as np
from sklearn.metrics import f1_score, recall_score, accuracy_score
import pandas as pd

# median seed per model (matched pairs)
MODELS = [
  ('vanilla_cnn','Vanilla CNN',2025,'from-scratch'),
  ('inception_v3','Inception V3',7,'pretrained'),
  ('convnext_tiny','ConvNeXt-Tiny',99,'pretrained'),
  ('swin_base','Swin-Base',123,'pretrained'),
  ('deit_base','DeiT-Base',123,'pretrained'),
]
CLASS_NAMES=['Mild','Moderate','No_DR','Proliferate_DR','Severe']
SEV=CLASS_NAMES.index('Severe'); PRO=CLASS_NAMES.index('Proliferate_DR')

In [2]:
# locate every _preds.npz (masked and unmasked) across attached inputs
pred_files={}
for r,_,fs in os.walk('/kaggle/input'):
    for x in fs:
        if x.endswith('_preds.npz'):
            pred_files[x]=os.path.join(r,x)
print('Found', len(pred_files), 'prediction files:')
for k in sorted(pred_files): print('  ',k)

Found 26 prediction files:
   convnext_tiny_masked_seed99_preds.npz
   convnext_tiny_seed123_preds.npz
   convnext_tiny_seed2025_preds.npz
   convnext_tiny_seed42_preds.npz
   convnext_tiny_seed7_preds.npz
   convnext_tiny_seed99_preds.npz
   deit_base_masked_seed123_preds.npz
   deit_base_seed123_preds.npz
   deit_base_seed2025_preds.npz
   deit_base_seed42_preds.npz
   inception_v3_masked_seed7_preds.npz
   inception_v3_seed123_preds.npz
   inception_v3_seed2025_preds.npz
   inception_v3_seed42_preds.npz
   inception_v3_seed7_preds.npz
   inception_v3_seed99_preds.npz
   swin_base_masked_seed123_preds.npz
   swin_base_seed123_preds.npz
   swin_base_seed2025_preds.npz
   swin_base_seed42_preds.npz
   vanilla_cnn_masked_seed2025_preds.npz
   vanilla_cnn_seed123_preds.npz
   vanilla_cnn_seed2025_preds.npz
   vanilla_cnn_seed42_preds.npz
   vanilla_cnn_seed7_preds.npz
   vanilla_cnn_seed99_preds.npz


In [3]:
def load_metrics(fname):
    d=np.load(pred_files[fname])
    y=d['labels']; p=d['preds']
    rec=recall_score(y,p,average=None,labels=range(5),zero_division=0)
    return dict(acc=accuracy_score(y,p), mf1=f1_score(y,p,average='macro',zero_division=0),
                sev=rec[SEV], pro=rec[PRO])

rows=[]
for slug,name,seed,typ in MODELS:
    um=f'{slug}_seed{seed}_preds.npz'
    mk=f'{slug}_masked_seed{seed}_preds.npz'
    if um not in pred_files:
        print('MISSING unmasked:',um); continue
    if mk not in pred_files:
        print('MISSING masked:',mk); continue
    u=load_metrics(um); m=load_metrics(mk)
    rows.append([name,typ,seed,
                 u['acc'],m['acc'],m['acc']-u['acc'],
                 u['mf1'],m['mf1'],m['mf1']-u['mf1'],
                 u['sev'],m['sev'],m['sev']-u['sev'],
                 u['pro'],m['pro'],m['pro']-u['pro']])
cols=['Model','Type','Seed','Acc_um','Acc_mk','dAcc','F1_um','F1_mk','dF1',
      'Sev_um','Sev_mk','dSev','Pro_um','Pro_mk','dPro']
df=pd.DataFrame(rows,columns=cols)
pd.set_option('display.width',200,'display.max_columns',20)
print(df.round(3).to_string(index=False))
df.round(4).to_csv('/kaggle/working/masked_vs_unmasked_metrics.csv',index=False)
print('\nSaved masked_vs_unmasked_metrics.csv')

        Model         Type  Seed  Acc_um  Acc_mk   dAcc  F1_um  F1_mk    dF1  Sev_um  Sev_mk  dSev  Pro_um  Pro_mk   dPro
  Vanilla CNN from-scratch  2025   0.767   0.616 -0.151  0.733  0.574 -0.159   0.448   0.598 0.149   0.591   0.012 -0.579
 Inception V3   pretrained     7   0.953   0.979  0.026  0.954  0.976  0.023   0.816   0.977 0.161   0.982   0.957 -0.024
ConvNeXt-Tiny   pretrained    99   0.828   0.922  0.093  0.834  0.921  0.087   0.828   0.966 0.138   0.537   0.793  0.256
    Swin-Base   pretrained   123   0.965   0.989  0.024  0.957  0.985  0.028   0.862   0.989 0.126   0.976   0.994  0.018
    DeiT-Base   pretrained   123   0.938   0.979  0.041  0.926  0.975  0.049   0.793   0.977 0.184   0.933   0.976  0.043

Saved masked_vs_unmasked_metrics.csv


## Summary view: the accuracy change by model type

In [4]:
print('ACCURACY change (masked - unmasked), by seed-matched pair:\n')
for _,row in df.iterrows():
    arrow='UP  ' if row['dAcc']>0 else 'DOWN'
    print(f"  {row['Model']:<15} ({row['Type']:<12}) {arrow} {row['dAcc']:+.3f}")
print()
print('PROLIFERATE recall change (the peripheral-disease signal):')
for _,row in df.iterrows():
    print(f"  {row['Model']:<15} {row['Pro_um']:.3f} -> {row['Pro_mk']:.3f}  ({row['dPro']:+.3f})")

ACCURACY change (masked - unmasked), by seed-matched pair:

  Vanilla CNN     (from-scratch) DOWN -0.151
  Inception V3    (pretrained  ) UP   +0.026
  ConvNeXt-Tiny   (pretrained  ) UP   +0.093
  Swin-Base       (pretrained  ) UP   +0.024
  DeiT-Base       (pretrained  ) UP   +0.041

PROLIFERATE recall change (the peripheral-disease signal):
  Vanilla CNN     0.591 -> 0.012  (-0.579)
  Inception V3    0.982 -> 0.957  (-0.024)
  ConvNeXt-Tiny   0.537 -> 0.793  (+0.256)
  Swin-Base       0.976 -> 0.994  (+0.018)
  DeiT-Base       0.933 -> 0.976  (+0.043)


## Notes for interpretation
- Single seed per model (matched pair), without-handling arm. Report as 'for the seed examined'.
- The direction (from-scratch hurt, pretrained helped) is consistent across all five, so robust; magnitudes carry single-seed uncertainty.
- Watch Proliferate recall for the from-scratch model — the peripheral-disease collapse is the mechanism.